# 第33版完整安装：3秒FPS、2小时运行

完整ZIP与本安装本上传同目录。停止旧程序后执行全部；进入TARGET重启内核，运行p11_start.ipynb。保留29版训练模型和配套硬件。


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, signal, stat, subprocess, sys, tempfile, zipfile

BASE = Path.cwd().resolve()
PACKAGE_NAME = 'p11_rgb_v9_ai_fps_20261101_33'
ZIP_SHA256 = '423609f84ddbdbfdee59a369d5fe39f938731ad59e46ebad292a4103c1c8f50d'
MANIFEST_SHA256 = '9b7bb4a50d757329162989babc9301ea213815646ecefe570cb134d02352b933'
RELEASE = 'p11-rgb-pspl-v9-ai-fps-20261101-33'

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def validated_parts(name):
    parts = PurePosixPath(name).parts
    if (not parts or parts[0] != PACKAGE_NAME or '..' in parts
            or '\\' in name or name.startswith('/') or any(':' in p for p in parts)):
        raise RuntimeError('Unsafe archive path: '+name)
    return parts[1:]

def install_full():
    # Accept WeChat's appended (1) filename when the complete bytes match.
    candidates = sorted(BASE.glob(PACKAGE_NAME+'*.zip'))
    archive = next((p for p in candidates if p.is_file() and digest(p) == ZIP_SHA256), None)
    if archive is None:
        raise RuntimeError('Upload the NEW full ZIP beside this notebook. No matching ZIP SHA256 found: '+str(BASE))
    with zipfile.ZipFile(archive) as z:
        entries = [info for info in z.infolist() if not info.is_dir()]
        names = [info.filename for info in entries]
        if len(names) != len(set(names)) or sum(i.file_size for i in entries) > 64*1024*1024:
            raise RuntimeError('Invalid archive inventory or uncompressed size')
        for info in z.infolist():
            validated_parts(info.filename)
            if stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('Symlinks are not allowed in this package')
        manifest_name = PACKAGE_NAME+'/release_manifest.json'
        data = z.read(manifest_name)
        for config_name in ('v98_ai_manifest.json','v99_ai_selftest.json'):
            config=json.loads(z.read(PACKAGE_NAME+'/'+config_name))
            if config.get('release_id')!=RELEASE:
                raise RuntimeError('AI/selftest metadata version mismatch: '+config_name)
        if hashlib.sha256(data).hexdigest() != MANIFEST_SHA256:
            raise RuntimeError('Release manifest SHA256 mismatch')
        manifest = json.loads(data)
        if manifest.get('release_id') != RELEASE:
            raise RuntimeError('Release ID mismatch')
        expected = manifest['sha256']
        if set(names) != {PACKAGE_NAME+'/'+name for name in expected} | {manifest_name}:
            raise RuntimeError('Archive has missing or extra files')
        required = {'system.bit','system.hwh','p11_start.ipynb','v102_ai_selection.py',
            'ai/plane.ncnn.bin','ai/plane.ncnn.param','v99_ai_selftest.py','v105_edges.cpp','v109_target_cycle.py','v110_unified.py','v111_recovery.py','v113_live_preview.py','v123_live_ui.py','v123_live_ui.png','v123_live_ui.json','pic78_cpu_check.py'}
        if not required.issubset(expected):
            raise RuntimeError('Required AI or hardware resources are missing')
        target = BASE/PACKAGE_NAME
        # Never overwrite an older/incomplete install, including this release.
        number = 2
        while target.exists():
            target = BASE/(PACKAGE_NAME+'_'+str(number)); number += 1
        stage = Path(tempfile.mkdtemp(prefix='_p11_flowfix_',dir=BASE)).resolve()
        if stage.parent != BASE or target.parent != BASE:
            raise RuntimeError('Installation directory escaped its parent')
        print('Verified ZIP. Installing %d files into %s'%(len(entries),target),flush=True)
        for info in entries:
            parts = validated_parts(info.filename)
            if not parts:raise RuntimeError('Missing archive filename')
            path = stage.joinpath(*parts).resolve()
            if not path.is_relative_to(stage):raise RuntimeError('Extraction path escaped staging')
            path.parent.mkdir(parents=True,exist_ok=True)
            h=hashlib.sha256()
            with z.open(info) as source, path.open('xb') as dest:
                for block in iter(lambda:source.read(1024*1024),b''):
                    dest.write(block);h.update(block)
            relative=PurePosixPath(*parts).as_posix()
            wanted=MANIFEST_SHA256 if relative=='release_manifest.json' else expected[relative]
            if h.hexdigest()!=wanted:raise RuntimeError('Extracted file mismatch: '+relative)
        for name,wanted in expected.items():
            if digest(stage/name)!=wanted:raise RuntimeError('Disk verification failed: '+name)
        if target.exists():raise RuntimeError('Target appeared during install; preserved staging: '+str(stage))
        stage.replace(target)
    (target/'install_full_receipt.json').write_text(json.dumps(dict(release_id=RELEASE,
        source_archive=str(archive),archive_sha256=ZIP_SHA256,complete_archive=True,
        old_directory_files_reused=0,all_files_verified=True,target=str(target)),indent=2)+'\n',encoding='utf-8')
    print('Installed:',target,flush=True)
    return target

TARGET = install_full()


In [ ]:
print('开始AI图片自检：使用包内图片，不打开相机或FPGA。',flush=True)
child=subprocess.Popen([sys.executable,'-u','-B',str(TARGET/'v99_ai_selftest.py')],cwd=str(TARGET),
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=dict(os.environ,PYTHONUNBUFFERED='1'))
try:
    for line in child.stdout:print(line,end='',flush=True)
    result=child.wait()
except KeyboardInterrupt:
    child.send_signal(signal.SIGINT)
    try:child.wait(4.)
    except subprocess.TimeoutExpired:child.kill();child.wait()
    raise
if result:
    report=TARGET/'ai_selftest_result.json'
    if report.is_file():print(report.read_text(encoding='utf-8'),flush=True)
    raise RuntimeError('安装已完成，AI自检未通过。请提供本单元输出及 '+str(report))
print('AI自检通过，继续执行下面的CPU检查。目标目录：',TARGET)


## CPU内核检查

上一单元通过后执行。本步骤不打开相机或FPGA。


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-B',str(TARGET/'pic78_cpu_check.py')],cwd=str(TARGET),check=True)
print('CPU检查通过。进入上述TARGET目录，重启内核，运行p11_start.ipynb。实时帧率及闪黑仍需板上测试。')
